In [ ]:
"""
Time-Delay Embeddings
Why use time-delay embedding?
We can highlight the difference between chaos and noisy dynamics.

Terminology
Manifold - a topological space that locally resembles Euclidean space near each point.
Phase space - a space where every possible state of a system is represented as a single point. Every degree of freedom or parameter of the system is represented as an axis of a multidimensional space.
Attractor - a set of states toward which a system tends to evolve to, for a wide variety of starting conditions of the system.
Isomorphism - Isomorphism is a structure-preserving mapping or morphism between two structures of the same type that can be reversed by an inverse mapping.
Diffeomorphism - A diffeomorphism is an isomorphism of differentiable manifolds. It is an invertible function that maps one differentiable manifold to another such that both the function and its inverse are continuously differentiable.
Lyapunov exponent of a dynamical system - A quantity that characterizes the exponential rate of separation of infinitesimally close trajectories.
Hankel Matrix - Each column in the Hankel Matrix is one data point in the delay coordinates (in the phase space). To put it simply, just delay the time series data and stack shifted copies of the data next to each other.

We typically start with the Lorentz system when testing algorithms for nonlinear dynamics. 
We use time-delay embedding to transform a sequence of partial observations into a higher-dimensional phase space that is topologically equivalent to the system's true, unobserved state space shown possible by Takens' Theorem.

Takens' Theorem:
Simply put, you can reconstruct a shadowed version of an original manifold simply by looking one of its time series projections, provided we choose tau and the no. of dimensions (m) appropriately.
Full state attractor constructed with varying dimensions are diffeomorphic with one another as well as the true full state.
Let the no. of dimensions of the full state coordinates be m. With Taken's theorem, we need the no. of dimensions in the phase space (n) to be at least 2m+1 (n>2m+1).

Due to the Picard-Lindelöf Theorem:
Simply put, as long as the system is smooth, the paths in phase space will not intersect as the states are deterministic and unique.
In theory, we need tau > 0 for the paths to not appear to merge or intersect, but practically, tau needs to be larger so that delayed measurements are distinct.

The Lorenz system:
A model of weather with 3 variables. Hence, it has 3D phase space. The Lorenz system never settles down and never exactly repeats itself, forming a Strange Attractor.
If we take a single time series variable, and plot the phase space, where the 3 axis are X(t), X(t-τ) and X(t-2τ), we can construct a shadowed version of the original butterfly manifold.
Each point in the 3 dimensional reconstruction can be thought of a time segment, with different points capturing different segments of history of variable X.
The recontrusted manifold is the library or collection of the historical behaviour of X.
Reconstruction preserves properties, such as topology of manifold and its Lyapunov exponents.
This is also a 1 to 1 mapping between the original manifold butterfly attractor and the reconstruction, allowing us to recover states of the original dynamic system by using legs of a single time series.
Given data in the full state coordinates, we can discover underlying dynamics with Sparse Identification of Nonlinear Dynamics (SINDy) with a form of sparse regression.

Neural Networks
We can use neural networks to approximate the full state coordinates with the delay coordinates. (to discover the actual differential equations that describe the system)
To start, we have state variables like z1, z2 ... and we build a huge library of potential terms like z1^2, sin(z1)... Then, with SINDy, we can find out which terms are important, as well as get the analytical function f(z).
SINDy also aims to minimize the residual error and maximize sparsity (trying to get equations to have as little terms as possible).
SINDy is giving the rate of change (velocity) of z. We cannot just run SINDy directly on the Hankel Matrix because its would give high-dimensional results but we want more sparse ones.

Taken's theorem says there is a mapping between time-delay coordinates (h) and the true hidden state (z), but it doesn't tell us what that mapping is.
Because the transformation is highly nonlinear, a Neural Network (specifically an Autoencoder) is the perfect tool to learn it. Singular Value Decomposition can be used first to perform a linear compression of the delay coordinates before feeding them into the neural network.
The autoencoder compresses the input space into a low and cleaner dimensional latent space z, supposedly, we should be able to reconstruct the actual phase state from the latent space.
SINDy and the autoencoder trains at the same time, and we compare them using loss functions that try to minimize the overlapping errors on both models simultaneously.
Loss functions:
1. The decoded output hbar must perfectly match the original input h
2. The derivatives calculated by the neural network's chain rule must match derivatives predicted by the SINDy equation (minimizing the difference between zdot and the SINDy zdotbar, as well as hdot and hdotbar).
3. Integrating SINDy model forward in time, the first dimension of that simulation is needs to match the original 1D input data. This makes the latent variables related to the actual physical measurements.
4. The network is continually punished for adding unnecessary mathematical terms. (Sparsity)

Problems:
Multiple different equations can validly explain the same 1D data as there are too many hyperparameters.
By trying to maximize sparsity, the shape of the latent attractor can slowly drift away from the true system.

Solution: 
The final discovered equation heavily depends on how the different loss functions are weighted. If we know certain rules about the system (e.g., energy must be conserved), we can add those information into the loss function force it to find the correct real-world physics.
"""